# Sorting

Adapted from Qualtran v0.7.0 commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import SVG, display
from examples.algorithms.sorting_catalog import sorting_example_graph, sorting_examples
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

examples = sorting_examples()

def show_signatures(group):
    display(SVG(field_signature_svg([(name, signature, {}) for name, signature, _ in group])))

def title(node):
    if isinstance(node, tuple):
        return f"{node[0]}({', '.join(map(str, node[1:]))})" if len(node) > 1 else node[0]
    return str(node)

def show_calls(example):
    graph = sorting_example_graph(example)
    display(SVG(render_call_graph_svg(export_call_graph_dot(
        graph.root, graph.direct_calls, title=title,
    ))))
    print("Source-declared leaves", graph.leaf_counts)

## `Comparator`
Compare and potentially swaps two n-bit numbers.

Implements $|a\rangle|b\rangle \mapsto |\min(a,b)\rangle|\max(a,b)\rangle|a>b\rangle$,

where $a$ and $b$ are n-qubit quantum registers. On output a and b are
swapped if a > b. Forms the base primitive for sorting.

#### Parameters
 - `bitsize`: value of $n$ (i.e. the inputs are $n$-bit numbers). 

#### Registers
 - `a`: A n-bit-sized input register (register a above).
 - `b`: A n-bit-sized input register (register b above).
 - `out`: A single bit output register which will store the result of the comparator. 

#### References
 - [Improved techniques for preparing eigenstates of fermionic Hamiltonians](https://www.nature.com/articles/s41534-018-0071-5). Fig. 1. in main text.


The comparator history is a logical output. The exact inverse consumes that history and clears it while restoring the inputs.

In [ ]:
comparators = examples[:2]

### Example Instances

In [ ]:
comparators[0]

In [ ]:
comparators[1]

#### Graphical Signature

In [ ]:
show_signatures(comparators)

### Call Graph

In [ ]:
show_calls(comparators[0])

## `BitonicSort`
Sort k n-bit integers in-place using a Bitonic sorting network.

For a given input list $x_1, x_2, \ldots, x_k$, applies the transform

$$
    |x_1, x_2, \ldots, x_k\rangle \mapsto |y_1, y_2, \ldots, y_k\rangle|\mathsf{junk}\rangle
$$

where $y_1, y_2, \ldots, y_k = \mathrm{sorted}(x_1, x_2, \ldots, x_k)$, and the junk register
stores the result of comparisons done during the sorting. Note that the `junk` register will
be entangled with the input list register.

Currently only supports $k$ being a power of two (#1090).

The bitonic sorting network requires $\frac{k}{2} \frac{\log{k} (1+\log{k})}{2}$ total comparisons,
and has depth $\frac{\log{k} (1+\log{k})}{2}$, when $k$ is a power of 2. Each comparison generates
one ancilla qubit that stores the result of the comparison, so the total size of `junk` register
equals the number of comparisons.

#### Parameters
 - `k`: Number of integers to sort.
 - `bitsize`: number of bits $n$ of each input number. 

#### Registers
 - `xs`: List of k integers we want to sort.
 - `junk`: the generated ancilla qubits of each comparison in the sorting network. 

#### References
 - [Improved techniques for preparing eigenstates of fermionic Hamiltonians](https://www.nature.com/articles/s41534-018-0071-5). Supporting Information Sec. II.


Native words are big-endian. Shared clean compare work is distinct from the logical comparison histories.

In [ ]:
sorters = examples[2:4]

### Example Instances

In [ ]:
sorters[0]

In [ ]:
sorters[1]

#### Graphical Signature

In [ ]:
show_signatures(sorters)

### Call Graph

In [ ]:
show_calls(sorters[0])

## `ParallelComparators`
Given k n-bit numbers, for each pair that is `offset` apart, compare and swap if needed to order them.

For each block of `2 * offset` numbers, apply a `Comparator` between each pair that is `offset` apart.
For an offset of $\delta$, this requires

$$
    \delta \left\lfloor \frac{k}{2\delta} \right\rfloor
    + \max((k\mod 2\delta) - \delta, 0)
$$

totals comparisons. The above expression is at most $k / 2$. Each comparison generates one ancilla qubit
which stores the result of comparsion, and these qubits are aggregated into the `junk` register.

This is used by `BitonicMerge` to apply parallel merges with offsets 1, 2, 4 and so on.

#### Parameters
 - `k`: size of the input list.
 - `offset`: compare numbers whose indices are offset apart.
 - `bitsize`: value of $n$ (i.e. the inputs are $n$-bit numbers). 

#### Registers
 - `xs`: input list of numbers.
 - `junk`: ancilla generated by comparators.


In [ ]:
parallel = examples[4]

### Example Instances

In [ ]:
parallel

#### Graphical Signature

In [ ]:
show_signatures([parallel])

### Call Graph

In [ ]:
show_calls(parallel)

## `BitonicMerge`
Merge two sorted sequences of n-bit integers.

Given two sorted lists of length `half_length`, merges them inplace into a single
sorted list.

Currently only supports `half_length` equal to a power of two (#1090).

If each half has length $k$, then the merge network uses $k (1+\log{k})$ comparisons
when $k$ is a power of 2. Each comparison generates one ancilla qubit which stores
the result of comparsion, and these qubits are aggregated into the `junk` register.

#### Parameters
 - `half_length`: Number of integers in each half
 - `bitsize`: value of $n$ (i.e. the inputs are $n$-bit numbers). 

#### Registers
 - `xs`: first input list of size `half_length`
 - `ys`: second input list of size `half_length`
 - `result`: merged output list of size `2 * half_length`
 - `junk`: ancilla generated by comparators.


In [ ]:
merge = examples[5]

### Example Instances

In [ ]:
merge

#### Graphical Signature

In [ ]:
show_signatures([merge])

### Call Graph

In [ ]:
show_calls(merge)

## Execute through append

The native example applies merge and sort through ordinary `circuit.append(op)`. It checks complex output amplitudes, restores the input under the exact inverse, and prints primitive counts separately from logical output histories.

In [ ]:
from examples.algorithms.register_bitonic_sort import main

main()